In [1]:
import random
import shutil
from pathlib import Path

CLEANED_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")
SAMPLE_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_sample_500")
SAMPLE_FOLDER.mkdir(exist_ok=True)

all_files = list(CLEANED_FOLDER.glob("*.yaml"))

random.seed(42)  # reproducible
sample = random.sample(all_files, 500)

for fp in sample:
    shutil.copy(fp, SAMPLE_FOLDER / fp.name)

print(f"Sampled : {len(sample)} files")
print(f"Output  : {SAMPLE_FOLDER}")

Sampled : 500 files
Output  : C:\Users\ensi\Desktop\python ai\ai_jobs_sample_500


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

os.environ["NVIDIA_API_KEY"] 

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)

response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-30b-a3b",
    messages=[
        {"role": "system", "content": "You are a precise information extractor. You only output valid JSON. No explanation, no markdown, no extra text."},
        {"role": "user", "content": "say hello"}
    ],
    max_tokens=2048,
    temperature=0,
    top_p=0.1,
    extra_body={"chat_template_kwargs": {"thinking": False}}  # ← disable reasoning trace
)

msg = response.choices[0].message
output = msg.content or msg.reasoning_content
print(output)

{
  "hashtags": [],
  "mentions": [],
  "text": "say hello"
}


In [11]:
from pathlib import Path

SAMPLE_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_sample_500")
files = list(SAMPLE_FOLDER.glob("*.yaml"))
print(f"Sample files: {len(files)}")

Sample files: 500


In [21]:
import json
import re
import time
import yaml
from pathlib import Path
from collections import defaultdict

# ── folders ───────────────────────────────────────────
SAMPLE_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_sample_500")
OUTPUT_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_extracted")
FAILED_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_failed")
OUTPUT_FOLDER.mkdir(exist_ok=True)
FAILED_FOLDER.mkdir(exist_ok=True)

# ── taxonomy ──────────────────────────────────────────
CATEGORIES = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]

# ── normalization map ─────────────────────────────────
NORMALIZATION = {
    # languages
    "python3"            : "Python",
    "python 3"           : "Python",
    "js"                 : "JavaScript",
    "javascript"         : "JavaScript",
    "typescript"         : "TypeScript",
    "ts"                 : "TypeScript",
    "golang"             : "Go",
    "c++"                : "C++",
    "cplusplus"          : "C++",
    # frontend
    "react"              : "React",
    "reactjs"            : "React",
    "react.js"           : "React",
    "react js"           : "React",
    "vuejs"              : "Vue.js",
    "vue"                : "Vue.js",
    "vue js"             : "Vue.js",
    "angular"            : "Angular",
    "angularjs"          : "Angular",
    "next.js"            : "Next.js",
    "nextjs"             : "Next.js",
    "next js"            : "Next.js",
    # backend
    "node"               : "Node.js",
    "nodejs"             : "Node.js",
    "node.js"            : "Node.js",
    "node js"            : "Node.js",
    "express"            : "Express",
    "expressjs"          : "Express",
    "express.js"         : "Express",
    "django"             : "Django",
    "flask"              : "Flask",
    "fastapi"            : "FastAPI",
    "fast api"           : "FastAPI",
    "spring"             : "Spring",
    "spring boot"        : "Spring Boot",
    # databases
    "postgres"           : "PostgreSQL",
    "postgresql"         : "PostgreSQL",
    "mongo"              : "MongoDB",
    "mongodb"            : "MongoDB",
    "mysql"              : "MySQL",
    "redis"              : "Redis",
    "elasticsearch"      : "Elasticsearch",
    "elastic search"     : "Elasticsearch",
    # devops / cloud
    "aws"                : "AWS",
    "amazon web services": "AWS",
    "gcp"                : "GCP",
    "google cloud"       : "GCP",
    "azure"              : "Azure",
    "microsoft azure"    : "Azure",
    "docker"             : "Docker",
    "kubernetes"         : "Kubernetes",
    "k8s"                : "Kubernetes",
    "terraform"          : "Terraform",
    "ci/cd"              : "CI/CD",
    "cicd"               : "CI/CD",
    "github actions"     : "GitHub Actions",
    "jenkins"            : "Jenkins",
    # ai / ml
    "pytorch"            : "PyTorch",
    "torch"              : "PyTorch",
    "tensorflow"         : "TensorFlow",
    "tf"                 : "TensorFlow",
    "huggingface"        : "HuggingFace",
    "hugging face"       : "HuggingFace",
    "transformers"       : "HuggingFace Transformers",
    "langchain"          : "LangChain",
    "lang chain"         : "LangChain",
    "openai"             : "OpenAI",
    "gpt"                : "GPT",
    "gpt-4"              : "GPT-4",
    "llm"                : "LLM",
    "llms"               : "LLM",
    # tools
    "git"                : "Git",
    "github"             : "GitHub",
    "gitlab"             : "GitLab",
    "jira"               : "Jira",
    "linux"              : "Linux",
    "vscode"             : "VS Code",
    "vs code"            : "VS Code",
}

def normalize_term(term: str) -> str:
    return NORMALIZATION.get(term.strip().lower(), term.strip())

# ── text fields ───────────────────────────────────────
TEXT_FIELDS = [
    "responsibilities",
    "basic_qualifications",
    "preferred_qualifications",
    "description",
    "summary",
]

# ── prompts ───────────────────────────────────────────
SYSTEM_PROMPT = """You are a precise tech stack extractor for job postings.
You only output valid JSON. No explanation, no markdown, no extra text.
You must strictly follow the output schema provided."""

def build_prompt(text: str) -> str:
    return f"""Extract technologies from this job posting text.

OUTPUT FORMAT (strict JSON, no extra fields):
{{
  "languages":    [],
  "frontend":     [],
  "backend":      [],
  "databases":    [],
  "devops_cloud": [],
  "ai_ml":        [],
  "tools":        []
}}

RULES:
- Only extract explicitly mentioned technologies
- Do NOT invent or infer technologies not in the text
- Do NOT add explanations or markdown
- If nothing found for a category, return empty list []

JOB TEXT:
{text}"""

# ── helpers ───────────────────────────────────────────
def extract_text(data: dict) -> str:
    lines = []
    for field in TEXT_FIELDS:
        for item in (data.get(field) or []):
            if item:
                lines.append(str(item).strip())
    return "\n".join(lines)

def clean_json_string(raw: str) -> str:
    raw = raw.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```\s*",     "", raw)
    raw = re.sub(r"\s*```$",     "", raw)
    return raw.strip()

def parse_response(raw: str) -> dict | None:
    if not raw:
        return None
    try:
        parsed = json.loads(clean_json_string(raw))
        if all(k in parsed for k in CATEGORIES):
            if all(isinstance(parsed[k], list) for k in CATEGORIES):
                return parsed
    except json.JSONDecodeError:
        pass
    return None

def normalize_result(result: dict) -> dict:
    normalized = {}
    for cat in CATEGORIES:
        seen  = set()
        clean = []
        for term in result.get(cat, []):
            if not isinstance(term, str) or not term.strip():
                continue
            normed = normalize_term(term)
            if normed.lower() not in seen:
                seen.add(normed.lower())
                clean.append(normed)
        normalized[cat] = clean
    return normalized

def call_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-nano-30b-a3b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt}
        ],
        max_tokens=2048,
        temperature=0,
        top_p=0.1,
        extra_body={"chat_template_kwargs": {"thinking": False}}
    )
    msg = response.choices[0].message
    raw = msg.content or msg.reasoning_content or ""

    # extract JSON block if model still reasoned
    if raw and not raw.strip().startswith("{"):
        match = re.search(r"\{[\s\S]*\}", raw)
        if match:
            return match.group(0)

    return raw

def exponential_backoff(attempt: int):
    time.sleep(2 ** (attempt - 1))  # 1s, 2s, 4s

# ── frequency tracker ─────────────────────────────────
freq = defaultdict(lambda: defaultdict(int))

# ── main loop ─────────────────────────────────────────
MAX_RETRIES  = 3
saved        = 0
failed       = 0
skipped      = 0
empty_result = 0

files = sorted(SAMPLE_FOLDER.glob("*.yaml"))

for i, fp in enumerate(files):

    out_path    = OUTPUT_FOLDER / fp.with_suffix(".json").name
    failed_path = FAILED_FOLDER / fp.with_suffix(".json").name

    if out_path.exists() or failed_path.exists():
        skipped += 1
        continue

    data = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))
    text = extract_text(data)

    if not text.strip():
        failed_path.write_text(
            json.dumps({"job_id": data.get("job_id"), "reason": "no text"}, indent=2),
            encoding="utf-8"
        )
        failed += 1
        continue

    prompt = build_prompt(text)
    result = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw    = call_llm(prompt)
            parsed = parse_response(raw)

            if parsed is not None:
                result = parsed
                break

            print(f"  [{i+1}/{len(files)}] {fp.name} — attempt {attempt} invalid JSON, retrying...")
            exponential_backoff(attempt)

        except Exception as e:
            print(f"  [{i+1}/{len(files)}] {fp.name} — attempt {attempt} error: {e}")
            exponential_backoff(attempt)

    if result is not None:
        result = normalize_result(result)
        result["job_id"] = str(data.get("job_id", ""))

        for cat in CATEGORIES:
            for term in result[cat]:
                freq[cat][term] += 1

        if all(len(result[cat]) == 0 for cat in CATEGORIES):
            empty_result += 1

        out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
        saved += 1

    else:
        failed_path.write_text(
            json.dumps({"job_id": str(data.get("job_id", "")), "reason": "max retries reached"}, indent=2),
            encoding="utf-8"
        )
        failed += 1

    if (i + 1) % 50 == 0:
        print(f"Progress {i+1}/{len(files)} — saved: {saved}, failed: {failed}, empty: {empty_result}, skipped: {skipped}")

# ── frequency report ──────────────────────────────────
print(f"\n✅ Done — saved: {saved}, failed: {failed}, empty: {empty_result}, skipped: {skipped}")
print("\n── Top terms per category ──")
for cat in CATEGORIES:
    top = sorted(freq[cat].items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\n{cat}:")
    for term, count in top:
        print(f"  {term:<30} {count}")

  [1/500] 6238063_Full_Stack_Engineer_Agentic_Ai.yaml — attempt 1 invalid JSON, retrying...
  [1/500] 6238063_Full_Stack_Engineer_Agentic_Ai.yaml — attempt 2 invalid JSON, retrying...
  [1/500] 6238063_Full_Stack_Engineer_Agentic_Ai.yaml — attempt 3 invalid JSON, retrying...
  [12/500] 6740948_Dba_Data_Engineer_Ai_First.yaml — attempt 1 invalid JSON, retrying...
  [15/500] 6945080_Ai_First_Senior_Qa_Engineer.yaml — attempt 1 invalid JSON, retrying...
  [15/500] 6945080_Ai_First_Senior_Qa_Engineer.yaml — attempt 2 invalid JSON, retrying...
  [15/500] 6945080_Ai_First_Senior_Qa_Engineer.yaml — attempt 3 invalid JSON, retrying...
  [22/500] 7096270_Software_Engineer_Ai.yaml — attempt 1 invalid JSON, retrying...
  [22/500] 7096270_Software_Engineer_Ai.yaml — attempt 2 invalid JSON, retrying...
  [39/500] 7309014_Software_Engineer_2.yaml — attempt 1 invalid JSON, retrying...
  [39/500] 7309014_Software_Engineer_2.yaml — attempt 2 invalid JSON, retrying...
  [39/500] 7309014_Software_Enginee

In [4]:
import json
from pathlib import Path
from collections import defaultdict

CATEGORIES = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]
OUTPUT_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_extracted")

freq = defaultdict(lambda: defaultdict(int))

for fp in OUTPUT_FOLDER.glob("*.json"):
    data = json.loads(fp.read_text(encoding="utf-8"))
    for cat in CATEGORIES:
        for term in data.get(cat, []):
            if term and term != data.get("job_id"):
                freq[cat][term] += 1

print("── Full term list per category ──")
for cat in CATEGORIES:
    terms = sorted(freq[cat].items(), key=lambda x: x[1], reverse=True)
    print(f"\n{cat} ({len(terms)} unique terms):")
    for term, count in terms:
        print(f"  {term:<35} {count}")

── Full term list per category ──

languages (58 unique terms):
  Python                              283
  Java                                71
  C++                                 62
  TypeScript                          54
  Go                                  49
  JavaScript                          44
  SQL                                 43
  C#                                  23
  Rust                                15
  Bash                                12
  C/C++                               11
  Scala                               11
  R                                   7
  Node.js                             6
  Swift                               5
  .NET                                5
  Kotlin                              5
  Ruby                                4
  HTML                                4
  CSS                                 4
  C                                   4
  python                              3
  Perl                                3
  P

In [3]:
import json
from pathlib import Path
from collections import defaultdict

OUTPUT_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_extracted")

freq = defaultdict(lambda: defaultdict(int))

for fp in OUTPUT_FOLDER.glob("*.json"):
    data = json.loads(fp.read_text(encoding="utf-8"))
    for cat in CATEGORIES:
        for term in data.get(cat, []):
            if term and term != data.get("job_id"):
                freq[cat][term] += 1

print("── Full term list per category ──")
for cat in CATEGORIES:
    terms = sorted(freq[cat].items(), key=lambda x: x[1], reverse=True)
    print(f"\n{cat} ({len(terms)} unique terms):")
    for term, count in terms:
        print(f"  {term:<35} {count}")

── Full term list per category ──

languages (58 unique terms):
  Python                              283
  Java                                71
  C++                                 62
  TypeScript                          54
  Go                                  49
  JavaScript                          44
  SQL                                 43
  C#                                  23
  Rust                                15
  Bash                                12
  C/C++                               11
  Scala                               11
  R                                   7
  Node.js                             6
  Swift                               5
  .NET                                5
  Kotlin                              5
  Ruby                                4
  HTML                                4
  CSS                                 4
  C                                   4
  python                              3
  Perl                                3
  P

In [4]:
# This is your validated TECH_DICTIONARY built from real frequency data
# One canonical name per term, one category per term, noise removed

TECH_DICTIONARY = {

    # ── LANGUAGES ─────────────────────────────────────
    "Python"         : "languages",
    "Java"           : "languages",
    "JavaScript"     : "languages",
    "TypeScript"     : "languages",
    "Go"             : "languages",
    "C\\+\\+"        : "languages",
    "C#"             : "languages",
    "Rust"           : "languages",
    "Bash"           : "languages",
    "SQL"            : "languages",
    "Scala"          : "languages",
    "Ruby"           : "languages",
    "Swift"          : "languages",
    "Kotlin"         : "languages",
    "R"              : "languages",
    "Perl"           : "languages",
    "PowerShell"     : "languages",
    "MATLAB"         : "languages",
    "Solidity"       : "languages",
    "Apex"           : "languages",

    # ── FRONTEND ──────────────────────────────────────
    "React"                    : "frontend",
    "Next\\.js"                : "frontend",
    "Angular"                  : "frontend",
    "Vue\\.js"                 : "frontend",
    "Flutter"                  : "frontend",
    "React Native"             : "frontend",
    "HTML"                     : "frontend",
    "CSS"                      : "frontend",
    "Svelte"                   : "frontend",
    "Tailwind CSS"             : "frontend",
    "TailwindCSS"              : "frontend",
    "WebGL"                    : "frontend",
    "Three\\.js"               : "frontend",
    "WebGPU"                   : "frontend",
    "Material UI"              : "frontend",
    "Styled Components"        : "frontend",
    "Storybook"                : "frontend",
    "SolidJS"                  : "frontend",
    "Streamlit"                : "frontend",
    "Lightning Web Components" : "frontend",
    "SwiftUI"                  : "frontend",

    # ── BACKEND ───────────────────────────────────────
    "Node\\.js"      : "backend",
    "FastAPI"        : "backend",
    "Flask"          : "backend",
    "Django"         : "backend",
    "Spring Boot"    : "backend",
    "Spring"         : "backend",
    "Express"        : "backend",
    "gRPC"           : "backend",
    "GraphQL"        : "backend",
    "Kafka"          : "backend",
    "RabbitMQ"       : "backend",
    "Celery"         : "backend",
    "FastAPI"        : "backend",
    "NestJS"         : "backend",
    "Ruby on Rails"  : "backend",
    "Rails"          : "backend",
    "WebSockets"     : "backend",
    "Flink"          : "backend",

    # ── DATABASES ─────────────────────────────────────
    "PostgreSQL"        : "databases",
    "MySQL"             : "databases",
    "MongoDB"           : "databases",
    "Redis"             : "databases",
    "Snowflake"         : "databases",
    "BigQuery"          : "databases",
    "Databricks"        : "databases",
    "Pinecone"          : "databases",
    "Weaviate"          : "databases",
    "Elasticsearch"     : "databases",
    "DynamoDB"          : "databases",
    "Cassandra"         : "databases",
    "SQLite"            : "databases",
    "Chroma"            : "databases",
    "ChromaDB"          : "databases",
    "pgvector"          : "databases",
    "Milvus"            : "databases",
    "OpenSearch"        : "databases",
    "Redshift"          : "databases",
    "RDS"               : "databases",
    "DocumentDB"        : "databases",
    "Firestore"         : "databases",
    "Qdrant"            : "databases",
    "Neo4j"             : "databases",
    "FAISS"             : "databases",
    "ClickHouse"        : "databases",
    "DuckDB"            : "databases",
    "Aurora"            : "databases",
    "Cosmos DB"         : "databases",
    "SQL Server"        : "databases",
    "Delta Lake"        : "databases",
    "Supabase"          : "databases",

    # ── DEVOPS / CLOUD ────────────────────────────────
    "AWS"               : "devops_cloud",
    "Azure"             : "devops_cloud",
    "GCP"               : "devops_cloud",
    "Kubernetes"        : "devops_cloud",
    "Docker"            : "devops_cloud",
    "Terraform"         : "devops_cloud",
    "CI/CD"             : "devops_cloud",
    "GitHub Actions"    : "devops_cloud",
    "Jenkins"           : "devops_cloud",
    "Helm"              : "devops_cloud",
    "Ansible"           : "devops_cloud",
    "ArgoCD"            : "devops_cloud",
    "Pulumi"            : "devops_cloud",
    "CloudFormation"    : "devops_cloud",
    "Prometheus"        : "devops_cloud",
    "Grafana"           : "devops_cloud",
    "SageMaker"         : "devops_cloud",
    "AWS Bedrock"       : "devops_cloud",
    "Vertex AI"         : "devops_cloud",
    "Azure DevOps"      : "devops_cloud",
    "OpenShift"         : "devops_cloud",
    "Kubeflow"          : "devops_cloud",
    "GitLab CI"         : "devops_cloud",
    "CircleCI"          : "devops_cloud",
    "Vercel"            : "devops_cloud",
    "Datadog"           : "devops_cloud",
    "S3"                : "devops_cloud",
    "Lambda"            : "devops_cloud",
    "EC2"               : "devops_cloud",
    "ECS"               : "devops_cloud",
    "VMware"            : "devops_cloud",
    "Cloudflare"        : "devops_cloud",
    "Heroku"            : "devops_cloud",
    "OpenTelemetry"     : "devops_cloud",
    "Ray"               : "devops_cloud",

    # ── AI / ML ───────────────────────────────────────
    "PyTorch"               : "ai_ml",
    "TensorFlow"            : "ai_ml",
    "LangChain"             : "ai_ml",
    "LangGraph"             : "ai_ml",
    "LlamaIndex"            : "ai_ml",
    "OpenAI"                : "ai_ml",
    "HuggingFace"           : "ai_ml",
    "RAG"                   : "ai_ml",
    "LLM"                   : "ai_ml",
    "GPT-4"                 : "ai_ml",
    "GPT"                   : "ai_ml",
    "Claude"                : "ai_ml",
    "Gemini"                : "ai_ml",
    "Mistral"               : "ai_ml",
    "Llama"                 : "ai_ml",
    "MLflow"                : "ai_ml",
    "Weights & Biases"      : "ai_ml",
    "scikit-learn"          : "ai_ml",
    "XGBoost"               : "ai_ml",
    "CUDA"                  : "ai_ml",
    "JAX"                   : "ai_ml",
    "CrewAI"                : "ai_ml",
    "AutoGen"               : "ai_ml",
    "DSPy"                  : "ai_ml",
    "vLLM"                  : "ai_ml",
    "TensorRT"              : "ai_ml",
    "DeepSpeed"             : "ai_ml",
    "ONNX"                  : "ai_ml",
    "Keras"                 : "ai_ml",
    "BERT"                  : "ai_ml",
    "Anthropic"             : "ai_ml",
    "LangSmith"             : "ai_ml",
    "Agentforce"            : "ai_ml",
    "Semantic Kernel"       : "ai_ml",
    "Azure OpenAI"          : "ai_ml",
    "Amazon Bedrock"        : "ai_ml",
    "SGLang"                : "ai_ml",
    "RLHF"                  : "ai_ml",
    "LoRA"                  : "ai_ml",
    "QLoRA"                 : "ai_ml",
    "PEFT"                  : "ai_ml",
    "spaCy"                 : "ai_ml",
    "NumPy"                 : "ai_ml",
    "Pandas"                : "ai_ml",
    "OpenCV"                : "ai_ml",
    "Stable Diffusion"      : "ai_ml",
    "MCP"                   : "ai_ml",
    "A2A"                   : "ai_ml",
    "Triton"                : "ai_ml",
    "Google ADK"            : "ai_ml",
    "TensorRT-LLM"          : "ai_ml",
    "Hugging Face"          : "ai_ml",

    # ── TOOLS ─────────────────────────────────────────
    "Git"               : "tools",
    "GitHub"            : "tools",
    "GitLab"            : "tools",
    "Jira"              : "tools",
    "Linux"             : "tools",
    "VS Code"           : "tools",
    "Cursor"            : "tools",
    "Spark"             : "tools",
    "Airflow"           : "tools",
    "Kafka"             : "tools",
    "Salesforce"        : "tools",
    "Tableau"           : "tools",
    "Power BI"          : "tools",
    "dbt"               : "tools",
    "Jupyter"           : "tools",
    "Pytest"            : "tools",
    "Selenium"          : "tools",
    "Playwright"        : "tools",
    "Confluence"        : "tools",
    "Slack"             : "tools",
    "Notion"            : "tools",
    "Zapier"            : "tools",
    "n8n"               : "tools",
    "HubSpot"           : "tools",
    "ServiceNow"        : "tools",
    "Workday"           : "tools",
    "Splunk"            : "tools",
    "Datadog"           : "tools",
    "Grafana"           : "tools",
    "Hadoop"            : "tools",
    "Flink"             : "tools",
    "PySpark"           : "tools",
    "Dagster"           : "tools",
    "Prefect"           : "tools",
    "GitHub Copilot"    : "tools",
    "Claude Code"       : "tools",
    "Windsurf"          : "tools",
    "Make"              : "tools",
    "Pydantic"          : "tools",
    "Okta"              : "tools",
    "Figma"             : "tools",
    "SAP"               : "tools",
    "Looker"            : "tools",
    "Sentry"            : "tools",
    "Postman"           : "tools",
    "ArgoCD"            : "tools",
    "Ansible"           : "tools",
}

print(f"Total terms in dictionary: {len(TECH_DICTIONARY)}")
print(f"Per category:")
from collections import Counter
counts = Counter(TECH_DICTIONARY.values())
for cat in CATEGORIES:
    print(f"  {cat:<20} {counts[cat]}")

Total terms in dictionary: 218
Per category:
  languages            20
  frontend             21
  backend              15
  databases            32
  devops_cloud         31
  ai_ml                52
  tools                47


In [5]:
import json
import yaml
import re
from pathlib import Path

CATEGORIES    = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]
SAMPLE_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_sample_500")
OUTPUT_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_extracted")

TEXT_FIELDS = ["responsibilities", "basic_qualifications", "preferred_qualifications", "description", "summary"]

def extract_text(data: dict) -> str:
    lines = []
    for field in TEXT_FIELDS:
        for item in (data.get(field) or []):
            if item:
                lines.append(str(item).strip())
    return "\n".join(lines)

TECH_DICTIONARY = {
    "Python": "languages", "Java": "languages", "JavaScript": "languages",
    "TypeScript": "languages", "Go": "languages", "C\\+\\+": "languages",
    "C#": "languages", "Rust": "languages", "Bash": "languages",
    "SQL": "languages", "Scala": "languages", "Ruby": "languages",
    "Swift": "languages", "Kotlin": "languages", "Perl": "languages",
    "PowerShell": "languages", "MATLAB": "languages", "Solidity": "languages",
    "Apex": "languages",
    "React": "frontend", "Next\\.js": "frontend", "Angular": "frontend",
    "Vue\\.js": "frontend", "Flutter": "frontend", "React Native": "frontend",
    "HTML": "frontend", "CSS": "frontend", "Svelte": "frontend",
    "Tailwind CSS": "frontend", "TailwindCSS": "frontend", "WebGL": "frontend",
    "Three\\.js": "frontend", "WebGPU": "frontend", "Material UI": "frontend",
    "Styled Components": "frontend", "Storybook": "frontend", "SolidJS": "frontend",
    "Streamlit": "frontend", "Lightning Web Components": "frontend", "SwiftUI": "frontend",
    "Node\\.js": "backend", "FastAPI": "backend", "Flask": "backend",
    "Django": "backend", "Spring Boot": "backend", "Spring": "backend",
    "Express": "backend", "gRPC": "backend", "GraphQL": "backend",
    "Kafka": "backend", "RabbitMQ": "backend", "Celery": "backend",
    "NestJS": "backend", "Ruby on Rails": "backend", "Rails": "backend",
    "WebSockets": "backend", "Flink": "backend",
    "PostgreSQL": "databases", "MySQL": "databases", "MongoDB": "databases",
    "Redis": "databases", "Snowflake": "databases", "BigQuery": "databases",
    "Databricks": "databases", "Pinecone": "databases", "Weaviate": "databases",
    "Elasticsearch": "databases", "DynamoDB": "databases", "Cassandra": "databases",
    "SQLite": "databases", "Chroma": "databases", "ChromaDB": "databases",
    "pgvector": "databases", "Milvus": "databases", "OpenSearch": "databases",
    "Redshift": "databases", "RDS": "databases", "DocumentDB": "databases",
    "Firestore": "databases", "Qdrant": "databases", "Neo4j": "databases",
    "FAISS": "databases", "ClickHouse": "databases", "DuckDB": "databases",
    "Aurora": "databases", "Cosmos DB": "databases", "SQL Server": "databases",
    "Delta Lake": "databases", "Supabase": "databases",
    "AWS": "devops_cloud", "Azure": "devops_cloud", "GCP": "devops_cloud",
    "Kubernetes": "devops_cloud", "Docker": "devops_cloud", "Terraform": "devops_cloud",
    "CI/CD": "devops_cloud", "GitHub Actions": "devops_cloud", "Jenkins": "devops_cloud",
    "Helm": "devops_cloud", "Ansible": "devops_cloud", "ArgoCD": "devops_cloud",
    "Pulumi": "devops_cloud", "CloudFormation": "devops_cloud", "Prometheus": "devops_cloud",
    "Grafana": "devops_cloud", "SageMaker": "devops_cloud", "AWS Bedrock": "devops_cloud",
    "Vertex AI": "devops_cloud", "Azure DevOps": "devops_cloud", "OpenShift": "devops_cloud",
    "Kubeflow": "devops_cloud", "GitLab CI": "devops_cloud", "CircleCI": "devops_cloud",
    "Vercel": "devops_cloud", "Datadog": "devops_cloud", "S3": "devops_cloud",
    "Lambda": "devops_cloud", "EC2": "devops_cloud", "ECS": "devops_cloud",
    "VMware": "devops_cloud", "Cloudflare": "devops_cloud", "Heroku": "devops_cloud",
    "OpenTelemetry": "devops_cloud", "Ray": "devops_cloud",
    "PyTorch": "ai_ml", "TensorFlow": "ai_ml", "LangChain": "ai_ml",
    "LangGraph": "ai_ml", "LlamaIndex": "ai_ml", "OpenAI": "ai_ml",
    "HuggingFace": "ai_ml", "RAG": "ai_ml", "LLM": "ai_ml",
    "GPT-4": "ai_ml", "GPT": "ai_ml", "Claude": "ai_ml",
    "Gemini": "ai_ml", "Mistral": "ai_ml", "Llama": "ai_ml",
    "MLflow": "ai_ml", "Weights & Biases": "ai_ml", "scikit-learn": "ai_ml",
    "XGBoost": "ai_ml", "CUDA": "ai_ml", "JAX": "ai_ml",
    "CrewAI": "ai_ml", "AutoGen": "ai_ml", "DSPy": "ai_ml",
    "vLLM": "ai_ml", "TensorRT": "ai_ml", "DeepSpeed": "ai_ml",
    "ONNX": "ai_ml", "Keras": "ai_ml", "BERT": "ai_ml",
    "Anthropic": "ai_ml", "LangSmith": "ai_ml", "Agentforce": "ai_ml",
    "Semantic Kernel": "ai_ml", "Azure OpenAI": "ai_ml", "Amazon Bedrock": "ai_ml",
    "SGLang": "ai_ml", "RLHF": "ai_ml", "LoRA": "ai_ml",
    "QLoRA": "ai_ml", "PEFT": "ai_ml", "spaCy": "ai_ml",
    "NumPy": "ai_ml", "Pandas": "ai_ml", "OpenCV": "ai_ml",
    "Stable Diffusion": "ai_ml", "MCP": "ai_ml", "A2A": "ai_ml",
    "Triton": "ai_ml", "Google ADK": "ai_ml", "TensorRT-LLM": "ai_ml",
    "Hugging Face": "ai_ml",
    "Git": "tools", "GitHub": "tools", "GitLab": "tools",
    "Jira": "tools", "Linux": "tools", "VS Code": "tools",
    "Cursor": "tools", "Spark": "tools", "Airflow": "tools",
    "Salesforce": "tools", "Tableau": "tools", "Power BI": "tools",
    "dbt": "tools", "Jupyter": "tools", "Pytest": "tools",
    "Selenium": "tools", "Playwright": "tools", "Confluence": "tools",
    "Slack": "tools", "Notion": "tools", "Zapier": "tools",
    "n8n": "tools", "HubSpot": "tools", "ServiceNow": "tools",
    "Workday": "tools", "Splunk": "tools", "Hadoop": "tools",
    "PySpark": "tools", "Dagster": "tools", "Prefect": "tools",
    "GitHub Copilot": "tools", "Claude Code": "tools", "Windsurf": "tools",
    "Make": "tools", "Pydantic": "tools", "Okta": "tools",
    "Figma": "tools", "SAP": "tools", "Looker": "tools",
    "Sentry": "tools", "Postman": "tools",
}

sorted_dict = sorted(TECH_DICTIONARY.items(), key=lambda x: len(x[0]), reverse=True)
COMPILED_PATTERNS = []
for term, category in sorted_dict:
    try:
        pattern = re.compile(r'(?<![a-zA-Z0-9])' + term + r'(?![a-zA-Z0-9])', re.IGNORECASE)
        canonical = term.replace("\\", "")
        COMPILED_PATTERNS.append((pattern, canonical, category))
    except re.error as e:
        print(f"Bad pattern: {term} → {e}")

def regex_extract(text: str) -> dict:
    result = {cat: [] for cat in CATEGORIES}
    seen   = set()
    for pattern, canonical, category in COMPILED_PATTERNS:
        if pattern.search(text):
            key = canonical.lower()
            if key not in seen:
                seen.add(key)
                result[category].append(canonical)
    return result

# ── compare regex vs llm on 10 files ─────────────────
files = sorted(SAMPLE_FOLDER.glob("*.yaml"))[:10]

for fp in files:
    data         = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))
    text         = extract_text(data)
    regex_result = regex_extract(text)
    llm_path     = OUTPUT_FOLDER / fp.with_suffix(".json").name
    llm_result   = json.loads(llm_path.read_text(encoding="utf-8")) if llm_path.exists() else None

    print(f"\n{'─'*60}")
    print(f"FILE: {fp.name}")
    for cat in CATEGORIES:
        regex_terms = set(regex_result.get(cat, []))
        llm_terms   = set(llm_result.get(cat, [])) if llm_result else set()
        only_regex  = regex_terms - llm_terms
        only_llm    = llm_terms - regex_terms
        both        = regex_terms & llm_terms
        if regex_terms or llm_terms:
            print(f"\n  {cat}:")
            if both:        print(f"    ✓ both   : {sorted(both)}")
            if only_regex:  print(f"    + regex  : {sorted(only_regex)}")
            if only_llm:    print(f"    + llm    : {sorted(only_llm)}")


────────────────────────────────────────────────────────────
FILE: 6238063_Full_Stack_Engineer_Agentic_Ai.yaml

  languages:
    + regex  : ['Go', 'Python', 'TypeScript']

  frontend:
    + regex  : ['React', 'React Native']

  backend:
    + regex  : ['Node.js']

  databases:
    + regex  : ['BigQuery']

  devops_cloud:
    + regex  : ['AWS', 'Azure', 'GCP', 'Kubernetes', 'SageMaker']

  ai_ml:
    + regex  : ['LLM', 'MCP']

  tools:
    + regex  : ['Airflow']

────────────────────────────────────────────────────────────
FILE: 6298916_Senior_Ai_Ml_Engineer.yaml

  languages:
    ✓ both   : ['Python', 'SQL']

  devops_cloud:
    ✓ both   : ['AWS', 'Azure']

  ai_ml:
    ✓ both   : ['LLM']
    + llm    : ['deep learning', 'foundational models', 'machine learning', 'natural language processing', 'natural language understanding']

────────────────────────────────────────────────────────────
FILE: 6362093_Senior_Site_Reliability_Engineer_Ai_Studio_Inference_Platform.yaml

  languages:
   

In [11]:
# terms to always drop from LLM output — too generic
NOISE_TERMS = {
    "ai", "ml", "machine learning", "deep learning", "artificial intelligence",
    "natural language processing", "natural language understanding", "nlp",
    "agentic systems", "agentic workflows", "agentic ai", "agentic frameworks",
    "foundation models", "foundational models", "large language models",
    "large language model", "generative ai", "gen ai", "genai",
    "fine-tuning", "finetuning", "prompt engineering", "prompt tuning",
    "llm inference", "llm optimization", "llm orchestration",
    "agent tooling", "eval frameworks", "evaluation frameworks",
    "ai models", "ai stack", "ai/llm systems", "ai features", "ai tools",
    "ai agents", "agents", "agent frameworks", "orchestration frameworks",
    "ml models", "ml ops", "llmops", "mlops", "llm apis",
    "embeddings", "vector search", "semantic search", "similarity search",
    "retrieval-augmented generation", "retrieval augmented generation",
    "rag pipelines", "rag systems", "rag frameworks",
    "model evaluation", "model fine-tuning", "model serving",
    "computer vision", "speech", "vision", "multimodal",
    "distributed systems", "cloud", "containerization", "serverless",
    "rest", "rest apis", "restful", "soap", "apis", "json", "yaml",
    "microservices", "devops", "devsecops", "gitops",
    "data pipelines", "data lake", "data warehouses",
    "llvms", "vlas", "foundation model",
}

print(f"Noise terms defined: {len(NOISE_TERMS)}")

Noise terms defined: 80


In [ ]:
import json
import re
import time
import yaml
from pathlib import Path
from collections import defaultdict
from openai import OpenAI
import os
from dotenv import load_doten

# ── config ────────────────────────────────────────────
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)

CLEANED_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")
OUTPUT_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\final_data_cleaned")
FAILED_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\final_data_failed")
OUTPUT_FOLDER.mkdir(exist_ok=True)
FAILED_FOLDER.mkdir(exist_ok=True)

CATEGORIES  = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]
TEXT_FIELDS = ["responsibilities", "basic_qualifications", "preferred_qualifications", "description", "summary"]

# ── noise blocklist ───────────────────────────────────
NOISE_TERMS = {
    "ai", "ml", "machine learning", "deep learning", "artificial intelligence",
    "natural language processing", "natural language understanding", "nlp",
    "agentic systems", "agentic workflows", "agentic ai", "agentic frameworks",
    "foundation models", "foundational models", "large language models",
    "large language model", "generative ai", "gen ai", "genai",
    "fine-tuning", "finetuning", "prompt engineering", "prompt tuning",
    "llm inference", "llm optimization", "llm orchestration",
    "agent tooling", "eval frameworks", "evaluation frameworks",
    "ai models", "ai stack", "ai/llm systems", "ai features", "ai tools",
    "ai agents", "agents", "agent frameworks", "orchestration frameworks",
    "ml models", "ml ops", "llmops", "mlops", "llm apis",
    "embeddings", "vector search", "semantic search", "similarity search",
    "retrieval-augmented generation", "retrieval augmented generation",
    "rag pipelines", "rag systems", "rag frameworks",
    "model evaluation", "model fine-tuning", "model serving",
    "computer vision", "speech", "vision", "multimodal",
    "distributed systems", "cloud", "containerization", "serverless",
    "rest", "rest apis", "restful", "soap", "apis", "json", "yaml",
    "microservices", "devops", "devsecops", "gitops",
    "data pipelines", "data lake", "data warehouses",
    "llvms", "vlas", "foundation model",
}

# ── normalization ─────────────────────────────────────
NORMALIZATION = {
    "python3": "Python", "python 3": "Python",
    "js": "JavaScript", "golang": "Go",
    "node": "Node.js", "nodejs": "Node.js", "node js": "Node.js",
    "reactjs": "React", "react.js": "React", "react js": "React",
    "vuejs": "Vue.js", "vue": "Vue.js",
    "nextjs": "Next.js", "next js": "Next.js",
    "postgres": "PostgreSQL", "postgresql": "PostgreSQL",
    "mongo": "MongoDB", "mongodb": "MongoDB",
    "huggingface": "HuggingFace", "hugging face": "HuggingFace",
    "langchain": "LangChain", "lang chain": "LangChain",
    "llm": "LLM", "llms": "LLM",
    "gpt-4": "GPT-4", "gpt4": "GPT-4",
    "aws": "AWS", "gcp": "GCP",
    "k8s": "Kubernetes",
    "ci/cd": "CI/CD", "cicd": "CI/CD",
    "github actions": "GitHub Actions",
    "pytorch": "PyTorch", "tensorflow": "TensorFlow",
    "scikit-learn": "scikit-learn", "sklearn": "scikit-learn",
    "vs code": "VS Code", "vscode": "VS Code",
}

def normalize_term(term: str) -> str:
    return NORMALIZATION.get(term.strip().lower(), term.strip())

# ── tech dictionary ───────────────────────────────────
TECH_DICTIONARY = {
    "Python": "languages", "Java": "languages", "JavaScript": "languages",
    "TypeScript": "languages", "Go": "languages", "C\\+\\+": "languages",
    "C#": "languages", "Rust": "languages", "Bash": "languages",
    "SQL": "languages", "Scala": "languages", "Ruby": "languages",
    "Swift": "languages", "Kotlin": "languages", "Perl": "languages",
    "PowerShell": "languages", "MATLAB": "languages", "Solidity": "languages",
    "Apex": "languages",
    "React": "frontend", "Next\\.js": "frontend", "Angular": "frontend",
    "Vue\\.js": "frontend", "Flutter": "frontend", "React Native": "frontend",
    "HTML": "frontend", "CSS": "frontend", "Svelte": "frontend",
    "Tailwind CSS": "frontend", "TailwindCSS": "frontend", "WebGL": "frontend",
    "Three\\.js": "frontend", "WebGPU": "frontend", "Material UI": "frontend",
    "Styled Components": "frontend", "Storybook": "frontend", "SolidJS": "frontend",
    "Streamlit": "frontend", "Lightning Web Components": "frontend", "SwiftUI": "frontend",
    "Node\\.js": "backend", "FastAPI": "backend", "Flask": "backend",
    "Django": "backend", "Spring Boot": "backend", "Spring": "backend",
    "Express": "backend", "gRPC": "backend", "GraphQL": "backend",
    "Kafka": "backend", "RabbitMQ": "backend", "Celery": "backend",
    "NestJS": "backend", "Ruby on Rails": "backend", "Rails": "backend",
    "WebSockets": "backend", "Flink": "backend",
    "PostgreSQL": "databases", "MySQL": "databases", "MongoDB": "databases",
    "Redis": "databases", "Snowflake": "databases", "BigQuery": "databases",
    "Databricks": "databases", "Pinecone": "databases", "Weaviate": "databases",
    "Elasticsearch": "databases", "DynamoDB": "databases", "Cassandra": "databases",
    "SQLite": "databases", "Chroma": "databases", "ChromaDB": "databases",
    "pgvector": "databases", "Milvus": "databases", "OpenSearch": "databases",
    "Redshift": "databases", "RDS": "databases", "DocumentDB": "databases",
    "Firestore": "databases", "Qdrant": "databases", "Neo4j": "databases",
    "FAISS": "databases", "ClickHouse": "databases", "DuckDB": "databases",
    "Aurora": "databases", "Cosmos DB": "databases", "SQL Server": "databases",
    "Delta Lake": "databases", "Supabase": "databases",
    "AWS": "devops_cloud", "Azure": "devops_cloud", "GCP": "devops_cloud",
    "Kubernetes": "devops_cloud", "Docker": "devops_cloud", "Terraform": "devops_cloud",
    "CI/CD": "devops_cloud", "GitHub Actions": "devops_cloud", "Jenkins": "devops_cloud",
    "Helm": "devops_cloud", "Ansible": "devops_cloud", "ArgoCD": "devops_cloud",
    "Pulumi": "devops_cloud", "CloudFormation": "devops_cloud", "Prometheus": "devops_cloud",
    "Grafana": "devops_cloud", "SageMaker": "devops_cloud", "AWS Bedrock": "devops_cloud",
    "Vertex AI": "devops_cloud", "Azure DevOps": "devops_cloud", "OpenShift": "devops_cloud",
    "Kubeflow": "devops_cloud", "GitLab CI": "devops_cloud", "CircleCI": "devops_cloud",
    "Vercel": "devops_cloud", "Datadog": "devops_cloud", "S3": "devops_cloud",
    "Lambda": "devops_cloud", "EC2": "devops_cloud", "ECS": "devops_cloud",
    "VMware": "devops_cloud", "Cloudflare": "devops_cloud", "Heroku": "devops_cloud",
    "OpenTelemetry": "devops_cloud", "Ray": "devops_cloud",
    "PyTorch": "ai_ml", "TensorFlow": "ai_ml", "LangChain": "ai_ml",
    "LangGraph": "ai_ml", "LlamaIndex": "ai_ml", "OpenAI": "ai_ml",
    "HuggingFace": "ai_ml", "RAG": "ai_ml", "LLM": "ai_ml",
    "GPT-4": "ai_ml", "GPT": "ai_ml", "Claude": "ai_ml",
    "Gemini": "ai_ml", "Mistral": "ai_ml", "Llama": "ai_ml",
    "MLflow": "ai_ml", "Weights & Biases": "ai_ml", "scikit-learn": "ai_ml",
    "XGBoost": "ai_ml", "CUDA": "ai_ml", "JAX": "ai_ml",
    "CrewAI": "ai_ml", "AutoGen": "ai_ml", "DSPy": "ai_ml",
    "vLLM": "ai_ml", "TensorRT": "ai_ml", "DeepSpeed": "ai_ml",
    "ONNX": "ai_ml", "Keras": "ai_ml", "BERT": "ai_ml",
    "Anthropic": "ai_ml", "LangSmith": "ai_ml", "Agentforce": "ai_ml",
    "Semantic Kernel": "ai_ml", "Azure OpenAI": "ai_ml", "Amazon Bedrock": "ai_ml",
    "SGLang": "ai_ml", "RLHF": "ai_ml", "LoRA": "ai_ml",
    "QLoRA": "ai_ml", "PEFT": "ai_ml", "spaCy": "ai_ml",
    "NumPy": "ai_ml", "Pandas": "ai_ml", "OpenCV": "ai_ml",
    "Stable Diffusion": "ai_ml", "MCP": "ai_ml", "A2A": "ai_ml",
    "Triton": "ai_ml", "Google ADK": "ai_ml", "TensorRT-LLM": "ai_ml",
    "Hugging Face": "ai_ml",
    "Git": "tools", "GitHub": "tools", "GitLab": "tools",
    "Jira": "tools", "Linux": "tools", "VS Code": "tools",
    "Cursor": "tools", "Spark": "tools", "Airflow": "tools",
    "Salesforce": "tools", "Tableau": "tools", "Power BI": "tools",
    "dbt": "tools", "Jupyter": "tools", "Pytest": "tools",
    "Selenium": "tools", "Playwright": "tools", "Confluence": "tools",
    "Slack": "tools", "Notion": "tools", "Zapier": "tools",
    "n8n": "tools", "HubSpot": "tools", "ServiceNow": "tools",
    "Workday": "tools", "Splunk": "tools", "Hadoop": "tools",
    "PySpark": "tools", "Dagster": "tools", "Prefect": "tools",
    "GitHub Copilot": "tools", "Claude Code": "tools", "Windsurf": "tools",
    "Make": "tools", "Pydantic": "tools", "Okta": "tools",
    "Figma": "tools", "SAP": "tools", "Looker": "tools",
    "Sentry": "tools", "Postman": "tools",
}

# ── compile regex patterns ────────────────────────────
sorted_dict = sorted(TECH_DICTIONARY.items(), key=lambda x: len(x[0]), reverse=True)
COMPILED_PATTERNS = []
for term, category in sorted_dict:
    try:
        pattern   = re.compile(r'(?<![a-zA-Z0-9])' + term + r'(?![a-zA-Z0-9])', re.IGNORECASE)
        canonical = term.replace("\\", "")
        COMPILED_PATTERNS.append((pattern, canonical, category))
    except re.error as e:
        print(f"Bad pattern: {term} → {e}")

print(f"Compiled {len(COMPILED_PATTERNS)} patterns")

# ── helpers ───────────────────────────────────────────
def extract_text(data: dict) -> str:
    lines = []
    for field in TEXT_FIELDS:
        for item in (data.get(field) or []):
            if item:
                lines.append(str(item).strip())
    return "\n".join(lines)

def regex_extract(text: str) -> dict:
    result = {cat: [] for cat in CATEGORIES}
    seen   = set()
    for pattern, canonical, category in COMPILED_PATTERNS:
        if pattern.search(text):
            key = canonical.lower()
            if key not in seen:
                seen.add(key)
                result[category].append(canonical)
    return result

def clean_json_string(raw: str) -> str:
    raw = raw.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```\s*",     "", raw)
    raw = re.sub(r"\s*```$",     "", raw)
    return raw.strip()

def parse_response(raw: str) -> dict | None:
    if not raw:
        return None
    try:
        parsed = json.loads(clean_json_string(raw))
        if all(k in parsed for k in CATEGORIES):
            if all(isinstance(parsed[k], list) for k in CATEGORIES):
                return parsed
    except json.JSONDecodeError:
        pass
    return None

def filter_noise(terms: list) -> list:
    return [t for t in terms if t.strip().lower() not in NOISE_TERMS and len(t.strip()) > 1]

def normalize_result(result: dict) -> dict:
    normalized = {}
    for cat in CATEGORIES:
        seen  = set()
        clean = []
        for term in result.get(cat, []):
            if not isinstance(term, str) or not term.strip():
                continue
            normed = normalize_term(term)
            if normed.lower() not in seen:
                seen.add(normed.lower())
                clean.append(normed)
        normalized[cat] = filter_noise(clean)
    return normalized

def merge_results(regex_result: dict, llm_result: dict) -> dict:
    """Regex wins on conflicts. LLM only adds what regex missed."""
    merged      = {}
    regex_seen  = set()

    # collect all canonical names regex found
    for cat in CATEGORIES:
        for term in regex_result.get(cat, []):
            regex_seen.add(term.lower())

    for cat in CATEGORIES:
        combined = list(regex_result.get(cat, []))
        seen     = {t.lower() for t in combined}

        for term in llm_result.get(cat, []):
            t_lower = term.lower()
            # only add if not already found by regex anywhere
            if t_lower not in seen and t_lower not in regex_seen:
                combined.append(term)
                seen.add(t_lower)

        merged[cat] = combined

    return merged

def build_llm_prompt(text: str, regex_result: dict) -> str:
    # show regex findings so LLM complements, not duplicates
    regex_summary = json.dumps(regex_result, indent=2)
    return f"""You are a tech stack extractor. Regex already found these technologies:
{regex_summary}

Find ADDITIONAL technologies explicitly mentioned in the job text that are NOT in the above list.
Only return technologies that are real, specific tools, frameworks, libraries, or platforms.
Do NOT return generic concepts like "machine learning", "AI", "deep learning", "microservices".

OUTPUT FORMAT (strict JSON, same schema, only NEW findings):
{{
  "languages":    [],
  "frontend":     [],
  "backend":      [],
  "databases":    [],
  "devops_cloud": [],
  "ai_ml":        [],
  "tools":        []
}}

JOB TEXT:
{text}"""

SYSTEM_PROMPT = """You are a precise tech stack extractor for job postings.
You only output valid JSON. No explanation, no markdown, no extra text."""

def call_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-nano-30b-a3b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt}
        ],
        max_tokens=3000,
        temperature=0,
        top_p=0.1,
        extra_body={"chat_template_kwargs": {"thinking": False}}
    )
    msg = response.choices[0].message
    raw = msg.content or msg.reasoning_content or ""
    if raw and not raw.strip().startswith("{"):
        match = re.search(r"\{[\s\S]*\}", raw)
        if match:
            return match.group(0)
    return raw

def exponential_backoff(attempt: int):
    time.sleep(2 ** (attempt - 1))

# ── main loop ─────────────────────────────────────────
MAX_RETRIES  = 3
saved        = 0
failed       = 0
skipped      = 0
empty_result = 0

files = sorted(CLEANED_FOLDER.glob("*.yaml"))

print(f"Total files to process: {len(files)}")

for i, fp in enumerate(files):

    out_path    = OUTPUT_FOLDER / fp.with_suffix(".json").name
    failed_path = FAILED_FOLDER / fp.with_suffix(".json").name

    # resume support
    if out_path.exists() or failed_path.exists():
        skipped += 1
        continue

    data = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))
    text = extract_text(data)
    job_id = str(data.get("job_id", ""))

    if not text.strip():
        failed_path.write_text(
            json.dumps({"job_id": job_id, "reason": "no text"}, indent=2),
            encoding="utf-8"
        )
        failed += 1
        continue

    # ── phase 1a: regex ───────────────────────────────
    regex_result = regex_extract(text)

    # ── phase 1b: llm fills gaps ──────────────────────
    llm_result = None
    prompt     = build_llm_prompt(text, regex_result)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw    = call_llm(prompt)
            parsed = parse_response(raw)
            if parsed is not None:
                llm_result = parsed
                break
            print(f"  [{i+1}/{len(files)}] attempt {attempt} invalid JSON, retrying...")
            exponential_backoff(attempt)
        except Exception as e:
            print(f"  [{i+1}/{len(files)}] attempt {attempt} error: {e}")
            exponential_backoff(attempt)

    # ── merge + normalize ─────────────────────────────
    if llm_result is not None:
        merged = merge_results(regex_result, llm_result)
    else:
        merged = regex_result  # fallback to regex only

    final = normalize_result(merged)

    if all(len(final[cat]) == 0 for cat in CATEGORIES):
        empty_result += 1

    # ── save ──────────────────────────────────────────
    final["job_id"]     = job_id
    final["job_title"]  = str(data.get("job_title", ""))
    final["date_scraped"] = str(data.get("date_scraped", ""))
    final["source_url"] = str(data.get("source_url", ""))

    if llm_result is None:
        failed_path.write_text(
            json.dumps({"job_id": job_id, "reason": "llm failed, regex only saved"}, indent=2),
            encoding="utf-8"
        )
        # still save regex result
        out_path.write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding="utf-8")
        failed += 1
    else:
        out_path.write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding="utf-8")
        saved += 1

    if (i + 1) % 100 == 0:
        print(f"Progress {i+1}/{len(files)} — saved: {saved}, failed: {failed}, empty: {empty_result}, skipped: {skipped}")

print(f"\n✅ Phase 1 Done — saved: {saved}, failed: {failed}, empty: {empty_result}, skipped: {skipped}")

Compiled 217 patterns
Total files to process: 2500

✅ Phase 1 Done — saved: 0, failed: 0, empty: 0, skipped: 2500


In [8]:
import json
import re
import time
import yaml
from pathlib import Path
from collections import defaultdict
CATEGORIES  = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]

# ── config ────────────────────────────────────────────
CLEANED_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")
OUTPUT_FOLDER   = Path(r"C:\Users\ensi\Desktop\python ai\final_data_cleaned")
CANDIDATE_FILE  = Path(r"C:\Users\ensi\Desktop\python ai\candidate_terms.json")

MAX_RETRIES = 3
saved_updated = 0
unchanged     = 0
failed        = 0

files = sorted(CLEANED_FOLDER.glob("*.yaml"))
print(f"Total files: {len(files)}")

# ── candidate tracker ─────────────────────────────────
# { term_lower: { "canonical": str, "category": str, "count": int, "seen_in": [job_ids] } }
candidates = defaultdict(lambda: {"canonical": "", "category": "", "count": 0, "seen_in": []})

for i, fp in enumerate(files):

    out_path = OUTPUT_FOLDER / fp.with_suffix(".json").name

    # load existing phase 1 result
    if not out_path.exists():
        failed += 1
        continue

    existing = json.loads(out_path.read_text(encoding="utf-8"))
    existing_terms = set()
    for cat in CATEGORIES:
        for t in existing.get(cat, []):
            existing_terms.add(t.lower())

    # load yaml
    data   = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))
    text   = extract_text(data)
    job_id = str(data.get("job_id", ""))

    if not text.strip():
        continue

    # ── re-run regex ──────────────────────────────────
    regex_result = regex_extract(text)

    # ── re-run llm ────────────────────────────────────
    llm_result = None
    prompt     = build_llm_prompt(text, regex_result)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw    = call_llm(prompt)
            parsed = parse_response(raw)
            if parsed is not None:
                llm_result = parsed
                break
            print(f"  [{i+1}/{len(files)}] attempt {attempt} invalid JSON, retrying...")
            exponential_backoff(attempt)
        except Exception as e:
            print(f"  [{i+1}/{len(files)}] attempt {attempt} error: {e}")
            exponential_backoff(attempt)

    # ── merge phase 2 results ─────────────────────────
    if llm_result is not None:
        merged = merge_results(regex_result, llm_result)
    else:
        merged = regex_result

    final_p2 = normalize_result(merged)

    # ── find new terms vs phase 1 ─────────────────────
    new_found = False
    for cat in CATEGORIES:
        for term in final_p2.get(cat, []):
            t_lower = term.lower()

            # already in dictionary → skip
            if any(t_lower == c.replace("\\", "").lower() for c in TECH_DICTIONARY):
                continue

            # already in phase 1 result → skip
            if t_lower in existing_terms:
                continue

            # new candidate found
            candidates[t_lower]["canonical"] = term
            candidates[t_lower]["category"]  = cat
            candidates[t_lower]["count"]     += 1
            if job_id not in candidates[t_lower]["seen_in"]:
                candidates[t_lower]["seen_in"].append(job_id)
            new_found = True

    # ── enrich existing file with new phase 2 terms ───
    enriched  = {cat: list(existing.get(cat, [])) for cat in CATEGORIES}
    enriched_seen = {t.lower() for cat in CATEGORIES for t in enriched[cat]}
    changed   = False

    for cat in CATEGORIES:
        for term in final_p2.get(cat, []):
            t_lower = term.lower()
            if t_lower not in enriched_seen:
                enriched[cat].append(term)
                enriched_seen.add(t_lower)
                changed = True

    if changed:
        # preserve metadata
        enriched["job_id"]      = existing.get("job_id", "")
        enriched["job_title"]   = existing.get("job_title", "")
        enriched["date_scraped"]= existing.get("date_scraped", "")
        enriched["source_url"]  = existing.get("source_url", "")
        out_path.write_text(json.dumps(enriched, indent=2, ensure_ascii=False), encoding="utf-8")
        saved_updated += 1
    else:
        unchanged += 1

    if (i + 1) % 100 == 0:
        print(f"Progress {i+1}/{len(files)} — updated: {saved_updated}, unchanged: {unchanged}, failed: {failed}, candidates so far: {len(candidates)}")

# ── save candidate terms sorted by frequency ──────────
sorted_candidates = dict(
    sorted(candidates.items(), key=lambda x: x[1]["count"], reverse=True)
)

CANDIDATE_FILE.write_text(
    json.dumps(sorted_candidates, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"\n✅ Phase 2 Done")
print(f"   Updated files  : {saved_updated}")
print(f"   Unchanged      : {unchanged}")
print(f"   Failed         : {failed}")
print(f"   New candidates : {len(candidates)}")
print(f"   Saved to       : {CANDIDATE_FILE}")

Total files: 2500
  [1/2500] attempt 1 invalid JSON, retrying...
  [1/2500] attempt 2 invalid JSON, retrying...
  [19/2500] attempt 1 invalid JSON, retrying...
  [33/2500] attempt 1 invalid JSON, retrying...
  [40/2500] attempt 1 invalid JSON, retrying...
  [43/2500] attempt 1 invalid JSON, retrying...
  [43/2500] attempt 2 invalid JSON, retrying...
  [60/2500] attempt 1 invalid JSON, retrying...
Progress 100/2500 — updated: 8, unchanged: 92, failed: 0, candidates so far: 10
  [102/2500] attempt 1 invalid JSON, retrying...
  [104/2500] attempt 1 invalid JSON, retrying...
  [104/2500] attempt 2 invalid JSON, retrying...
  [104/2500] attempt 3 invalid JSON, retrying...
  [120/2500] attempt 1 invalid JSON, retrying...
  [152/2500] attempt 1 invalid JSON, retrying...
  [159/2500] attempt 1 invalid JSON, retrying...
Progress 200/2500 — updated: 24, unchanged: 176, failed: 0, candidates so far: 40
  [226/2500] attempt 1 invalid JSON, retrying...
  [235/2500] attempt 1 invalid JSON, retrying.

In [9]:
import json
from pathlib import Path

CANDIDATE_FILE = Path(r"C:\Users\ensi\Desktop\python ai\candidate_terms.json")
candidates = json.loads(CANDIDATE_FILE.read_text(encoding="utf-8"))

print(f"Total candidates: {len(candidates)}")
print(f"\n── Top 50 by frequency ──")
print(f"{'Term':<35} {'Category':<15} {'Count':<8} {'Seen in'}")
print("─" * 75)

for term, info in list(candidates.items())[:50]:
    print(f"  {info['canonical']:<33} {info['category']:<15} {info['count']:<8} {len(info['seen_in'])} jobs")

Total candidates: 635

── Top 50 by frequency ──
Term                                Category        Count    Seen in
───────────────────────────────────────────────────────────────────────────
  vector databases                  databases       14       14 jobs
  Transformers                      ai_ml           13       13 jobs
  .NET                              backend         11       11 jobs
  Guardrails                        ai_ml           11       11 jobs
  VectorDBs                         databases       11       11 jobs
  ChatGPT                           ai_ml           8        8 jobs
  Copilot                           tools           5        5 jobs
  Azure Machine Learning            devops_cloud    5        5 jobs
  Google Cloud                      devops_cloud    5        5 jobs
  VectorDB                          databases       4        4 jobs
  Ray Serve                         ai_ml           4        4 jobs
  Codex                             ai_ml           4

In [10]:
# ── add new validated terms to dictionary ─────────────
NEW_TERMS = {
    # ai_ml
    "Transformers"          : "ai_ml",
    "Guardrails"            : "ai_ml",
    "ChatGPT"               : "ai_ml",
    "Ray Serve"             : "ai_ml",
    "Codex"                 : "ai_ml",

    # languages
    "\\.NET"                : "languages",

    # databases
    "Oracle"                : "databases",
    "Azure SQL"             : "databases",
    "Unity Catalog"         : "databases",

    # devops_cloud
    "Azure Machine Learning": "devops_cloud",
    "Azure AI Foundry"      : "devops_cloud",
    "EKS"                   : "devops_cloud",
    "CloudWatch"            : "devops_cloud",
    "Vertex AI Pipelines"   : "devops_cloud",
    "Azure Monitor"         : "devops_cloud",

    # backend
    "Pub/Sub"               : "backend",
    "OpenAPI"               : "backend",

    # tools
    "Jest"                  : "tools",
    "ROS"                   : "tools",
    "FPGA"                  : "tools",
}

# ── add normalizations ────────────────────────────────
NEW_NORMALIZATIONS = {
    "google cloud"          : "GCP",
    "azure ml"              : "Azure Machine Learning",
    "transformers"          : "HuggingFace Transformers",
    ".net core"             : ".NET",
    ".net framework"        : ".NET",
    "chatgpt"               : "ChatGPT",
    "codex"                 : "Codex",
    "guardrails"            : "Guardrails",
}

# ── update dictionaries ───────────────────────────────
TECH_DICTIONARY.update(NEW_TERMS)
NORMALIZATION.update(NEW_NORMALIZATIONS)

# ── recompile patterns ────────────────────────────────
sorted_dict = sorted(TECH_DICTIONARY.items(), key=lambda x: len(x[0]), reverse=True)
COMPILED_PATTERNS = []
for term, category in sorted_dict:
    try:
        pattern   = re.compile(r'(?<![a-zA-Z0-9])' + term + r'(?![a-zA-Z0-9])', re.IGNORECASE)
        canonical = term.replace("\\", "")
        COMPILED_PATTERNS.append((pattern, canonical, category))
    except re.error as e:
        print(f"Bad pattern: {term} → {e}")

print(f"Dictionary size  : {len(TECH_DICTIONARY)} terms")
print(f"Compiled patterns: {len(COMPILED_PATTERNS)}")
print(f"New terms added  : {len(NEW_TERMS)}")

Dictionary size  : 237 terms
Compiled patterns: 237
New terms added  : 20


In [11]:
import json
import yaml
from pathlib import Path

CLEANED_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")
OUTPUT_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\final_data_cleaned")

updated   = 0
unchanged = 0
failed    = 0

files = sorted(CLEANED_FOLDER.glob("*.yaml"))
print(f"Total files: {len(files)}")

for i, fp in enumerate(files):

    out_path = OUTPUT_FOLDER / fp.with_suffix(".json").name

    if not out_path.exists():
        failed += 1
        continue

    # load existing result
    existing = json.loads(out_path.read_text(encoding="utf-8"))
    existing_seen = {t.lower() for cat in CATEGORIES for t in existing.get(cat, [])}

    # load yaml and re-run regex with updated dictionary
    data = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))
    text = extract_text(data)

    if not text.strip():
        continue

    regex_result = regex_extract(text)

    # find new terms not in existing
    changed = False
    for cat in CATEGORIES:
        for term in regex_result.get(cat, []):
            if term.lower() not in existing_seen:
                existing[cat].append(term)
                existing_seen.add(term.lower())
                changed = True

    if changed:
        out_path.write_text(json.dumps(existing, indent=2, ensure_ascii=False), encoding="utf-8")
        updated += 1
    else:
        unchanged += 1

    if (i + 1) % 500 == 0:
        print(f"Progress {i+1}/{len(files)} — updated: {updated}, unchanged: {unchanged}")

print(f"\n✅ Done — updated: {updated}, unchanged: {unchanged}, failed: {failed}")

Total files: 2500
Progress 500/2500 — updated: 55, unchanged: 444
Progress 1000/2500 — updated: 138, unchanged: 858
Progress 2000/2500 — updated: 324, unchanged: 1668
Progress 2500/2500 — updated: 411, unchanged: 2078

✅ Done — updated: 411, unchanged: 2078, failed: 11


In [12]:
import json
from pathlib import Path
from collections import defaultdict

OUTPUT_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\final_data_cleaned")
CATEGORIES    = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]

files        = list(OUTPUT_FOLDER.glob("*.json"))
total        = len(files)
empty        = []
non_empty    = 0
cat_coverage = defaultdict(int)
term_counts  = []

# duplicate tracking
jobs_with_intra_dupes = 0
total_dupe_terms      = 0
dupe_examples         = []

# cross-category duplicate tracking
jobs_with_cross_dupes = 0
cross_dupe_examples   = []

for fp in files:
    data = json.loads(fp.read_text(encoding="utf-8"))
    job_id = data.get("job_id", fp.name)

    all_empty    = True
    all_terms    = []
    cross_seen   = {}  # term_lower → first category seen

    intra_dupes  = []
    cross_dupes  = []

    for cat in CATEGORIES:
        terms = data.get(cat, [])
        if terms:
            all_empty = False
            cat_coverage[cat] += 1

        # intra-category duplicates (same term twice in same category)
        seen_in_cat = set()
        for term in terms:
            t_lower = term.lower().strip()
            if t_lower in seen_in_cat:
                intra_dupes.append((cat, term))
                total_dupe_terms += 1
            seen_in_cat.add(t_lower)

        # cross-category duplicates (same term in multiple categories)
        for term in terms:
            t_lower = term.lower().strip()
            if t_lower in cross_seen:
                cross_dupes.append((term, cross_seen[t_lower], cat))
            else:
                cross_seen[t_lower] = cat

        all_terms.extend(terms)

    term_counts.append(len(all_terms))

    if all_empty:
        empty.append(fp.name)
    else:
        non_empty += 1

    if intra_dupes:
        jobs_with_intra_dupes += 1
        if len(dupe_examples) < 3:
            dupe_examples.append((job_id, intra_dupes[:3]))

    if cross_dupes:
        jobs_with_cross_dupes += 1
        if len(cross_dupe_examples) < 3:
            cross_dupe_examples.append((job_id, cross_dupes[:3]))

# ── report ────────────────────────────────────────────
print(f"Total JSON files  : {total}")
print(f"Non-empty         : {non_empty}")
print(f"Empty             : {len(empty)}")
print(f"Avg terms per job : {sum(term_counts)/len(term_counts):.1f}")
print(f"Max terms         : {max(term_counts)}")
print(f"Min terms         : {min(term_counts)}")

print(f"\n── Category coverage ──")
for cat in CATEGORIES:
    pct = cat_coverage[cat] / total * 100
    bar = "█" * int(pct / 5)
    print(f"  {cat:<20} {cat_coverage[cat]:>5} jobs  {pct:>5.1f}%  {bar}")

print(f"\n── Intra-category duplicates (same term twice in same category) ──")
print(f"  Jobs affected    : {jobs_with_intra_dupes}")
print(f"  Total dupe terms : {total_dupe_terms}")
if dupe_examples:
    print(f"  Examples:")
    for job_id, dupes in dupe_examples:
        print(f"    job {job_id}: {dupes}")

print(f"\n── Cross-category duplicates (same term in multiple categories) ──")
print(f"  Jobs affected    : {jobs_with_cross_dupes}")
if cross_dupe_examples:
    print(f"  Examples:")
    for job_id, dupes in cross_dupe_examples:
        print(f"    job {job_id}: {dupes}")

if empty:
    print(f"\n── Empty files ──")
    for name in empty[:20]:
        print(f"  {name}")
    if len(empty) > 20:
        print(f"  ... and {len(empty)-20} more")

Total JSON files  : 2489
Non-empty         : 2403
Empty             : 86
Avg terms per job : 10.9
Max terms         : 58
Min terms         : 0

── Category coverage ──
  languages             1942 jobs   78.0%  ███████████████
  frontend               413 jobs   16.6%  ███
  backend                535 jobs   21.5%  ████
  databases              705 jobs   28.3%  █████
  devops_cloud          1518 jobs   61.0%  ████████████
  ai_ml                 1857 jobs   74.6%  ██████████████
  tools                 1713 jobs   68.8%  █████████████

── Intra-category duplicates (same term twice in same category) ──
  Jobs affected    : 0
  Total dupe terms : 0

── Cross-category duplicates (same term in multiple categories) ──
  Jobs affected    : 1
  Examples:
    job 8189294: [('AWS API Gateway', 'backend', 'devops_cloud'), ('Azure API Management', 'backend', 'devops_cloud')]

── Empty files ──
  6371888_Qa_Engineer_Ai_And_Data.json
  6757443_Senior_Data_Platform_Developer_Generative_Ai.json
  69

In [13]:
import json
from pathlib import Path

OUTPUT_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\final_data_cleaned")
CATEGORIES    = ["languages", "frontend", "backend", "databases", "devops_cloud", "ai_ml", "tools"]

deleted = 0

for fp in list(OUTPUT_FOLDER.glob("*.json")):
    data      = json.loads(fp.read_text(encoding="utf-8"))
    all_empty = all(len(data.get(cat, [])) == 0 for cat in CATEGORIES)
    if all_empty:
        fp.unlink()
        deleted += 1

print(f"Deleted : {deleted}")
print(f"Remaining : {len(list(OUTPUT_FOLDER.glob('*.json')))}")

Deleted : 86
Remaining : 2403
